In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as T
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader, Subset
from tqdm.notebook import tqdm
import kagglehub
import os
import random
import cv2
from PIL import Image, ImageDraw
import numpy as np

# 1. DYNAMIC DEP ALIGNMENT
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "mediapipe"])
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

# Ensure task file is pulled down locally to Colab runtime
if not os.path.exists("hand_landmarker.task"):
    print("📡 Pulling detection engine weights...")
    subprocess.run(["curl", "-o", "hand_landmarker.task", "https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task"])

# SKELETON WIRING HARNESS
HAND_LINKS = [(0,1),(1,2),(2,3),(3,4),(0,5),(5,6),(6,7),(7,8),
              (5,9),(9,10),(10,11),(11,12),(9,13),(13,14),(14,15),
              (15,16),(13,17),(17,18),(18,19),(19,20),(0,17)]

def render_skeleton(lms, size=100):
    xs = [lm.x for lm in lms]
    ys = [lm.y for lm in lms]
    xmin, xmax = min(xs), max(xs)
    ymin, ymax = min(ys), max(ys)
    side = max((xmax-xmin), (ymax-ymin)) * 1.3
    midx, midy = (xmin+xmax)/2, (ymin+ymax)/2
    canvas = Image.new("RGB", (size, size), (0, 0, 0))
    draw = ImageDraw.Draw(canvas)
    pts = []
    for lm in lms:
        lx = (lm.x - midx) / side + 0.5
        ly = (lm.y - midy) / side + 0.5
        pts.append((int(lx * size), int(ly * size)))
    for s, e in HAND_LINKS:
        draw.line([pts[s], pts[e]], fill=(255, 255, 255), width=3)
    for pt in pts:
        draw.ellipse([pt[0]-2, pt[1]-2, pt[0]+2, pt[1]+2], fill=(255, 255, 255))
    return canvas

def main():
    print("💾 RESOLVING KAGGLE ASSETS...")
    path = kagglehub.dataset_download("grassknoted/asl-alphabet")
    root = os.path.join(path, "asl_alphabet_train/asl_alphabet_train")
    out_root = "/content/skeleton_dataset"
    os.makedirs(out_root, exist_ok=True)

    classes = sorted([d for d in os.listdir(root) if os.path.isdir(os.path.join(root, d))])

    # SPIN UP ULTRA-MODERN TASKS DETECTOR
    print("⚡ Waking up Tasks Engine...")
    base_options = python.BaseOptions(model_asset_path='hand_landmarker.task')
    options = vision.HandLandmarkerOptions(base_options=base_options, num_hands=1)
    detector = vision.HandLandmarker.create_from_options(options)

    print("🔥 STEP 1: Constructing Synthetic Geometry Space...")

    img_limit_per_class = 500

    for cls in tqdm(classes, desc="Encoding Shapes"):
        c_in_dir = os.path.join(root, cls)
        c_out_dir = os.path.join(out_root, cls)
        os.makedirs(c_out_dir, exist_ok=True)

        files = [f for f in os.listdir(c_in_dir) if f.endswith('.jpg')]
        random.shuffle(files)

        processed = 0
        for f in files:
            if processed >= img_limit_per_class: break
            full_p = os.path.join(c_in_dir, f)

            try:
                # Standard image load
                mp_image = mp.Image.create_from_file(full_p)
                res = detector.detect(mp_image)

                if res.hand_landmarks:
                    skel = render_skeleton(res.hand_landmarks[0], 100)
                    skel.save(os.path.join(c_out_dir, f"{processed}.jpg"))
                    processed += 1
            except: continue

    print(f"\n✅ SYNTHESIS READY: Output parked at {out_root}")

if __name__ == "__main__":
    main()


📡 Pulling detection engine weights...
💾 RESOLVING KAGGLE ASSETS...
Using Colab cache for faster access to the 'asl-alphabet' dataset.
⚡ Waking up Tasks Engine...
🔥 STEP 1: Constructing Synthetic Geometry Space...


Encoding Shapes:   0%|          | 0/29 [00:00<?, ?it/s]


✅ SYNTHESIS READY: Output parked at /content/skeleton_dataset
🔥 STEP 2: Igniting CNN Training on pure geometry...
🚀 Propelling weights on cuda...
🎯 Accuracy -> 4.00%
🎯 Accuracy -> 4.24%
🎯 Accuracy -> 4.33%
🎯 Accuracy -> 4.33%
🎯 Accuracy -> 4.43%
🎯 Accuracy -> 4.47%
🎯 Accuracy -> 4.52%
🎯 Accuracy -> 4.52%
🎯 Accuracy -> 4.38%
🎯 Accuracy -> 4.52%
🎯 Accuracy -> 4.24%
🎯 Accuracy -> 4.43%
🎯 Accuracy -> 4.47%
🎯 Accuracy -> 4.57%
🎯 Accuracy -> 4.47%

🏆 WE ARE LIVE. Download 'asl_cnn_skeleton.pth' and install it locally.


In [ ]:
# ==========================================================
# FORTIFIED ARCHITECTURE
# ==========================================================
class BulletproofPaperCNN(nn.Module):
    def __init__(self, num_classes=29):
        super().__init__()
        def blk(c_in, c_out, ks):
            return nn.Sequential(
                nn.Conv2d(c_in, c_out, kernel_size=ks, padding=ks//2, bias=False),
                nn.BatchNorm2d(c_out),
                nn.LeakyReLU(0.1, inplace=True),
                nn.MaxPool2d(kernel_size=2, stride=3)
            )
        self.feats = nn.Sequential(blk(3, 8, 19), blk(8, 16, 17), blk(16, 32, 15))
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.3),
            nn.Linear(32 * 4 * 4, num_classes)
        )
        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, nn.Conv2d) or isinstance(m, nn.Linear):
            nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='leaky_relu')

    def forward(self, x):
        return self.classifier(self.feats(x))

# ==========================================================
# FINAL BENCHMARK TRAINING
# ==========================================================
def run_ultimate_tuning():
    data_path = "/content/skeleton_dataset"
    print("🚀 Commencing Ultimate Geometry Pass...")

    # KEY FIX: Increased rotation to 45 degrees to cover that downward wrist slope!
    tfms = T.Compose([
        T.RandomRotation(45),
        T.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.85, 1.15)),
        # NEW: Perspective simulations camera viewing angles perfectly!
        T.RandomPerspective(distortion_scale=0.15, p=0.5),
        T.ToTensor(),
        T.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
    ])

    ds = ImageFolder(root=data_path, transform=tfms)
    indices = list(range(len(ds)))
    random.shuffle(indices)
    split = int(len(ds) * 0.85)

    train_dl = DataLoader(Subset(ds, indices[:split]), batch_size=64, shuffle=True, num_workers=2)
    val_dl   = DataLoader(Subset(ds, indices[split:]), batch_size=64, shuffle=False, num_workers=2)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = BulletproofPaperCNN(num_classes=len(ds.classes)).to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=0.002)

    epochs = 40
    best_acc = 0.0

    for epoch in range(epochs):
        model.train()
        for x, y in train_dl:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            criterion(model(x), y).backward()
            optimizer.step()

        model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for x, y in val_dl:
                x, y = x.to(device), y.to(device)
                _, p = torch.max(model(x).data, 1)
                total += y.size(0)
                correct += (p == y).sum().item()

        acc = 100 * correct / total
        print(f"🔥 Epoch {epoch+1} -> Accuracy: {acc:.2f}%")

        if acc > best_acc:
            best_acc = acc
            torch.save({
                'model_state': model.state_dict(),
                'classes': ds.classes,
                'val_acc': best_acc
            }, "asl_cnn_skeleton_ultimate.pth")

    print("\n🎯 MASTER FILE READY. Download 'asl_cnn_skeleton_ultimate.pth' and replace file in backend.")

if __name__ == "__main__":
    run_ultimate_tuning()


🚀 Commencing Ultimate Geometry Pass...
🔥 Epoch 1 -> Accuracy: 59.73%
🔥 Epoch 2 -> Accuracy: 75.82%
🔥 Epoch 3 -> Accuracy: 75.77%
🔥 Epoch 4 -> Accuracy: 81.01%
🔥 Epoch 5 -> Accuracy: 85.67%
🔥 Epoch 6 -> Accuracy: 84.91%
🔥 Epoch 7 -> Accuracy: 88.20%
🔥 Epoch 8 -> Accuracy: 86.48%
🔥 Epoch 9 -> Accuracy: 90.67%
🔥 Epoch 10 -> Accuracy: 90.15%
🔥 Epoch 11 -> Accuracy: 90.91%
🔥 Epoch 12 -> Accuracy: 84.34%
🔥 Epoch 13 -> Accuracy: 93.53%
🔥 Epoch 14 -> Accuracy: 92.48%
🔥 Epoch 15 -> Accuracy: 92.77%
🔥 Epoch 16 -> Accuracy: 92.81%
🔥 Epoch 17 -> Accuracy: 93.67%
🔥 Epoch 18 -> Accuracy: 91.62%
🔥 Epoch 19 -> Accuracy: 93.48%
🔥 Epoch 20 -> Accuracy: 93.38%
🔥 Epoch 21 -> Accuracy: 93.86%
🔥 Epoch 22 -> Accuracy: 92.72%
🔥 Epoch 23 -> Accuracy: 89.48%
🔥 Epoch 24 -> Accuracy: 94.00%
🔥 Epoch 25 -> Accuracy: 93.67%
🔥 Epoch 26 -> Accuracy: 92.91%
🔥 Epoch 27 -> Accuracy: 90.86%
🔥 Epoch 28 -> Accuracy: 93.76%
🔥 Epoch 29 -> Accuracy: 95.10%
🔥 Epoch 30 -> Accuracy: 94.62%
🔥 Epoch 31 -> Accuracy: 94.15%
🔥 Epoch 3

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as T
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader, Subset
from tqdm.notebook import tqdm
import kagglehub
import os
import random
import cv2
from PIL import Image, ImageDraw
import numpy as np
import subprocess, sys

# ==========================================================
# STAGE 0: GLOBAL RUNTIME PROVISIONING
# ==========================================================
print("🛠️ STAGE 0: Bootstrapping High-Performance Libraries...")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "mediapipe"])
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

if not os.path.exists("hand_landmarker.task"):
    print("📡 Pulling canonical detection engine weights...")
    subprocess.run(["curl", "-o", "hand_landmarker.task", "https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task"])

# Master Joint Connections List
HAND_LINKS = [(0,1),(1,2),(2,3),(3,4),(0,5),(5,6),(6,7),(7,8),
              (5,9),(9,10),(10,11),(11,12),(9,13),(13,14),(14,15),
              (15,16),(13,17),(17,18),(18,19),(19,20),(0,17)]

def render_skeleton(lms, size=100):
    """High-fidelity geometric drawing engine with consistent scale mapping"""
    xs = [lm.x for lm in lms]
    ys = [lm.y for lm in lms]
    xmin, xmax = min(xs), max(xs)
    ymin, ymax = min(ys), max(ys)
    side = max((xmax-xmin), (ymax-ymin)) * 1.3 # Standardized 30% padding buffer
    midx, midy = (xmin+xmax)/2, (ymin+ymax)/2
    canvas = Image.new("RGB", (size, size), (0, 0, 0))
    draw = ImageDraw.Draw(canvas)
    pts = []
    for lm in lms:
        lx = (lm.x - midx) / side + 0.5
        ly = (lm.y - midy) / side + 0.5
        pts.append((int(lx * size), int(ly * size)))
    for s, e in HAND_LINKS:
        draw.line([pts[s], pts[e]], fill=(255, 255, 255), width=3)
    for pt in pts:
        draw.ellipse([pt[0]-2, pt[1]-2, pt[0]+2, pt[1]+2], fill=(255, 255, 255))
    return canvas

# ==========================================================
# STAGE 1: THE FORTIFIED RESEARCH ARCHITECTURE
# ==========================================================
class FortifiedPaperCNN(nn.Module):
    def __init__(self, num_classes=29):
        super().__init__()
        def blk(c_in, c_out, ks):
            return nn.Sequential(
                nn.Conv2d(c_in, c_out, kernel_size=ks, padding=ks//2, bias=False),
                nn.BatchNorm2d(c_out), # System Stabilization Vaccine
                nn.LeakyReLU(0.1, inplace=True), # Prevents Zero-Gradient Node Collapse
                nn.MaxPool2d(kernel_size=2, stride=3)
            )
        self.feats = nn.Sequential(blk(3, 8, 19), blk(8, 16, 17), blk(16, 32, 15))
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(p=0.3),
            nn.Linear(32 * 4 * 4, num_classes)
        )
        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, nn.Conv2d) or isinstance(m, nn.Linear):
            # Essential Kaiming normal tuning for custom deep kernels
            nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='leaky_relu')

    def forward(self, x):
        return self.classifier(self.feats(x))

# ==========================================================
# STAGE 2: END-TO-END MASTER EXECUTION
# ==========================================================
def execute_pipeline():
    print("\n💾 STAGE 1: Fetching Source Data & Preparing Workspace...")
    path = kagglehub.dataset_download("grassknoted/asl-alphabet")
    src_root = os.path.join(path, "asl_alphabet_train/asl_alphabet_train")
    target_root = "/content/skeleton_dataset_final"
    os.makedirs(target_root, exist_ok=True)

    classes = sorted([d for d in os.listdir(src_root) if os.path.isdir(os.path.join(src_root, d))])

    print("⚡ Spinning up local Vision Graph Tracker...")
    base_opts = python.BaseOptions(model_asset_path='hand_landmarker.task')
    opts = vision.HandLandmarkerOptions(base_options=base_opts, num_hands=1)
    detector = vision.HandLandmarker.create_from_options(opts)

    print("🔥 STAGE 2: Synthesizing Infinite-Immunity Geometry Samples...")
    limit = 500

    for cls in tqdm(classes, desc="Synthesizing Artifacts"):
        c_in = os.path.join(src_root, cls)
        c_out = os.path.join(target_root, cls)
        os.makedirs(c_out, exist_ok=True)

        # Process images and skip past ones that were already converted locally
        files = sorted([f for f in os.listdir(c_in) if f.endswith('.jpg')])
        random.shuffle(files)

        done = 0
        for f in files:
            if done >= limit: break
            try:
                m_img = mp.Image.create_from_file(os.path.join(c_in, f))
                res = detector.detect(m_img)
                if res.hand_landmarks:
                    render_skeleton(res.hand_landmarks[0], 100).save(os.path.join(c_out, f"{done}.jpg"))
                    done += 1
            except: continue

    print(f"\n✅ Data Synthesis Finished: Found generated assets at {target_root}")

    # 🚀 ENGAGE FINAL MODEL COMPILATION
    print("🔥 STAGE 3: Launching Ultimate Hardware Propagation...")

    # The Advanced Augmentation Pipeline securing absolute camera-tilt immunity
    tfms = T.Compose([
        T.RandomRotation(45), # Critical fix: Hardens wrist slant variance
        T.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.85, 1.15)),
        T.RandomPerspective(distortion_scale=0.15, p=0.5), # Simulates phone holding angles
        T.ToTensor(),
        T.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
    ])

    ds = ImageFolder(root=target_root, transform=tfms)
    indices = list(range(len(ds)))
    random.shuffle(indices)
    split = int(len(ds) * 0.85)

    train_dl = DataLoader(Subset(ds, indices[:split]), batch_size=64, shuffle=True, num_workers=2)
    val_dl   = DataLoader(Subset(ds, indices[split:]), batch_size=64, shuffle=False, num_workers=2)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"🚀 Igniting fortified weights on GPU device: {device}")

    model = FortifiedPaperCNN(num_classes=len(classes)).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=0.002)

    epochs = 15
    best_val = 0.0

    for epoch in range(epochs):
        model.train()
        for x, y in train_dl:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            criterion(model(x), y).backward()
            optimizer.step()

        model.eval()
        hit, total = 0, 0
        with torch.no_grad():
            for x, y in val_dl:
                x, y = x.to(device), y.to(device)
                _, p = torch.max(model(x).data, 1)
                total += y.size(0)
                hit += (p == y).sum().item()

        curr_acc = 100 * hit / total
        print(f"🎯 Validation Round {epoch+1} -> Precise Score: {curr_acc:.2f}%")

        if curr_acc > best_val:
            best_val = curr_acc
            torch.save({
                'model_state': model.state_dict(),
                'classes': ds.classes,
                'val_acc': best_val
            }, "asl_cnn_skeleton_ultimate.pth")
            print("🏆 High Score Preserved!")

    print("\n🎯 MISSION MASTERED. The absolute 'asl_cnn_skeleton_ultimate.pth' has been produced.")
    print("Download now via the sidebar folder tab.")

if __name__ == "__main__":
    execute_pipeline()
